# test_environment

用于验证 Environment 的 round 级机制。


In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'src' / 'task_router_graph').exists()
)
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from task_router_graph.schema import Environment, ControllerAction, Task

print('模块加载完成')


In [ ]:
def make_trace(tag: str):
    return [
        ControllerAction(action_kind='observe', reason=f'{tag}-read', tool='read', args={'path': 'README.md'}, observation='ok'),
        ControllerAction(action_kind='generate_task', reason=f'{tag}-route', task_type='normal', task_content=f'{tag}-content'),
    ]

def make_task(content: str, status: str = 'pending', result: str = ''):
    return Task(type='normal', content=content, status=status, result=result)

env = Environment()
round1 = env.start_round(user_input="用户输入1")
round2 = env.start_round(user_input="用户输入2")
assert round1.round_id == 1 and round2.round_id == 2
print('round 创建成功')


In [ ]:
# round#1 写入两条 task
trace1 = make_trace("r1t1")
task1 = make_task("first", status="failed", result="need retry")
record11 = env.add_task(round_id=1, controller_trace=trace1, task=task1, reply="reply11")
record12 = env.add_task(round_id=1, controller_trace=make_trace("r1t2"), task=make_task("second", status="done", result="ok"), reply="reply12")
assert record11.task_id == 1 and record12.task_id == 2

# round#2 写入一条 task（task_id 在 round 内从 1 重新计数）
record21 = env.add_task(round_id=2, controller_trace=make_trace("r2t1"), task=make_task("third", status="done", result="ok"), reply="reply21")
assert record21.task_id == 1
print('按 round 写入成功')


In [ ]:
# 非法 round_id 应直接报错
try:
    env.add_task(round_id=99, controller_trace=[], task=make_task("x"), reply="x")
    raise AssertionError("期望 add_task 抛错，但未抛错")
except ValueError as e:
    assert "round_id not found" in str(e)
print('非法 round_id 报错机制通过')


In [ ]:
# 深拷贝验证
trace1[0].reason = "mutated-externally"
task1.status = "done"
task1.result = "mutated-externally"

stored = env.rounds[0].tasks[0]
assert stored.controller_trace[0].reason == "r1t1-read"
assert stored.task.status == "failed"
assert stored.task.result == "need retry"
print('深拷贝验证通过')


In [ ]:
text = env.show_environment(show_trace=True)
assert "round_count: 2" in text
assert "task_count: 3" in text
print(text)


In [ ]:
obs_ctx = env.build_observation_view(task_limit=None, include_trace=True)
obs = obs_ctx['tasks']
assert obs_ctx['cur_round'] == 2
assert len(obs) == 3
assert obs[0]['round_id'] == 1 and obs[0]['task_id'] == 1
assert obs[2]['round_id'] == 2 and obs[2]['task_id'] == 1

rounds_view = env.build_rounds_view(include_trace=False)
assert len(rounds_view) == 2
assert 'controller_trace' not in rounds_view[0]['tasks'][0]
obs_ctx, rounds_view


In [ ]:
print('Environment round 级机制测试全部通过')
